# Operational Screening of Wine Bottling Lots (Red + White)

*MLN601 Assessment 2 - Decision Tree Classification and CRISP-DM*

Design and Creative Technologies, Torrens University

- **Student:** Luis Guilherme de Barros Andrade Faria - A00187785
- **Subject Name:** Machine Learning
- **Subject Code:** MLN 601
- **Title:** Operational Screening of Wine Bottling Lots (Red + White)
- **Lecturer:** Dr. Kamran Shaukat
- **Assessment No.:** 2
- **Date:** July 2026

| Field | Value |
|---|---|
| Dataset | UCI Wine Quality - red + white combined (Cortez et al., 2009) |
| Target | Low (`quality < 6`) = 1; high (`quality >= 6`) = 0 |
| Required algorithm | Decision Tree - default, AUC-tuned and balanced variants |
| Sensitivity studies | Leakage-safe SMOTE and kernel SVM benchmark |
| Methodology | CRISP-DM |

Each row in the UCI dataset is one laboratory sample. In this project, I treat that single sample as standing in for a whole bottling lot, assuming the lot is homogeneous - I am not classifying individual bottles. The model's job is to flag lots with a higher risk of low sensory quality so the quality-control team can send them for extra tasting or testing. It is a decision-support tool, where a human always makes the final call, and no lot is ever released or rejected automatically.

Quality is a subjective, ordinal score, so every model is framed as an interpretable decision-support tool, not a replacement for expert tasting.

## 1. Business Understanding

A wine producer needs to release bottling lots efficiently,  without letting low-quality lots reach distribution. Each row in the dataset is one laboratory-tested wine sample, which here stands for one bottling lot. The business question is: 

> *Can a model flag the lots at higher risk of low sensory quality, so they are sent for extra tasting or testing before release?*

Following the brief, `quality < 6` is considered low (class 1) and `quality >= 6` is considered high (class 0). The model's output is a binary classification, where the positive class (1) indicates a lot that should be sent for extra tasting or testing.

Operationally, a false negative (a low-quality lot classified as high-quality) is more costly than a false positive (a high-quality lot classified as low-quality). The model is therefore a triage layer between laboratory testing and human tasting, and it is designed to be interpretable so that quality-control staff can understand the reasoning behind its predictions and make the final call - the model never releases or rejects a lot on its own.

Success is defined **before modelling**: in five-fold cross-validation (CV), the training data is divided into 5 parts, the model trains on 4 and validates on the 5th, rotating until every part has been used for validation. A candidate must achieve:
- ROC-AUC >= 0.75 - how well a model separates the two classes
- low-quality sensitivity >= 0.70 - how well a test or model finds all the positive cases
- high-quality specificity >= 0.70 - how well a test or model finds all the negative cases

Among candidates passing every gate, priority goes to sensitivity, then balanced accuracy, interpretability, and simplicity. These are screening targets for this assessment, not validated production service levels.

## 2. Data Understanding

### 2.1 Data acquisition and operational unit

| Item | Detail |
|---|---|
| Source | UCI Machine Learning Repository - Wine Quality (id 186) |
| Link | https://archive.ics.uci.edu/dataset/186/wine+quality |
| Files | `winequality-red.csv` and `winequality-white.csv` |
| Acquisition | Local UCI copies with direct-URL fallback |
| Raw rows | 1,599 red + 4,898 white = 6,497 |
| Operational interpretation | One row is a proxy for one representative lot sample |

The source contains physicochemical tests and an expert sensory score for Portuguese *vinho verde* (Cortez et al., 2009). It has no `batch_id`, production date or release decision, so performance here demonstrates the model's ability to separate low and high quality and technical feasibility, not proven lot-level safety or return on investment. Red and white files are combined with `wine_type` (red = 1, white = 0).

### 2.2 Variables

| Variable group | Type and units | Role |
|---|---|---|
| Fixed, volatile and citric acidity; residual sugar; chlorides; sulphates | Continuous, g/dm3 | Laboratory predictors |
| Free and total sulfur dioxide | Continuous, mg/dm3 | Laboratory predictors |
| Density | Continuous, g/cm3 | Laboratory predictor |
| pH | Continuous, pH scale | Laboratory predictor |
| Alcohol | Continuous, % volume | Laboratory predictor |
| `wine_type` | Engineered binary | Red/white context |
| `quality` | Ordinal integer, source score 0-10 | Source target, excluded from predictors |
| `quality_label` | Engineered binary | Model target: low = 1, high = 0 |

### 2.3 Environment Setup and Data Collection


To keep results reproducible, this section pins the runtime (library versions, RANDOM_STATE = 42) and loads the red and white datasets from a local copy when available, falling back to the UCI repository. A wine_type flag (red = 1, white = 0) is added before combining them into a single dataframe of 6,497 rows.

In [ ]:
import os
import sys
import warnings
from importlib.metadata import version as package_version

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, balanced_accuracy_score,
                             make_scorer)

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except ImportError as exc:
    raise ImportError(
        "Install the Assessment2 requirements first: "
        "python -m pip install -r requirements.txt"
    ) from exc

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

runtime = {
    "Python": sys.version.split()[0],
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "scikit-learn": package_version("scikit-learn"),
    "imbalanced-learn": package_version("imbalanced-learn"),
    "shap": package_version("shap"),
}
pd.Series(runtime, name="version").to_frame()

In [ ]:
from pathlib import Path

# Resolve folders whether execution starts from notebook/ or the assessment root.
NB_DIR = Path.cwd()
BASE_DIR = NB_DIR.parent if NB_DIR.name == "notebook" else NB_DIR
OUTPUT_DIR = BASE_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

UCI_BASE = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
            "wine-quality/")


def load_wine(colour):
    fname = "winequality-%s.csv" % colour
    candidates = [BASE_DIR / "dataset" / fname,
                  NB_DIR / "dataset" / fname,
                  NB_DIR / fname]
    local = next((path for path in candidates if path.exists()), None)
    if local is not None:
        print("Loading local dataset:", local.name)
        return pd.read_csv(local, sep=";")
    print("Loading UCI dataset:", fname)
    return pd.read_csv(UCI_BASE + fname, sep=";")


red = load_wine("red")
white = load_wine("white")
red["wine_type"] = 1
white["wine_type"] = 0
df = pd.concat([red, white], ignore_index=True)

print("Red:", red.shape, "| White:", white.shape, "| Combined:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
source_features = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol",
]
expected_columns = source_features + ["quality", "wine_type"]
numeric_values = df[expected_columns].to_numpy()
duplicate_rows = int(df.duplicated().sum())

# IQR flags support an explicit outlier decision; they do not automatically imply errors.
q1 = df[source_features].quantile(0.25)
q3 = df[source_features].quantile(0.75)
iqr = q3 - q1
lower_bounds = q1 - 1.5 * iqr
upper_bounds = q3 + 1.5 * iqr
outlier_flags = (df[source_features].lt(lower_bounds) |
                 df[source_features].gt(upper_bounds))
rows_with_iqr_flags = int(outlier_flags.any(axis=1).sum())
outlier_summary = pd.DataFrame({
    "lower_iqr_bound": lower_bounds,
    "upper_iqr_bound": upper_bounds,
    "flagged_values": outlier_flags.sum(),
    "observed_min": df[source_features].min(),
    "observed_max": df[source_features].max(),
}).sort_values("flagged_values", ascending=False)

hard_checks = {
    "Expected schema": list(df.columns) == expected_columns,
    "All columns numeric": all(pd.api.types.is_numeric_dtype(df[c]) for c in expected_columns),
    "No missing values": int(df.isna().sum().sum()) == 0,
    "All values finite": bool(np.isfinite(numeric_values).all()),
    "No negative laboratory values": bool((df[source_features].to_numpy() >= 0).all()),
    "Quality is integer-valued": bool(np.allclose(df["quality"], df["quality"].astype(int))),
    "Quality within documented 0-10 range": bool(df["quality"].between(0, 10).all()),
    "Free SO2 does not exceed total SO2": bool(
        (df["free sulfur dioxide"] <= df["total sulfur dioxide"]).all()
    ),
}
assert all(hard_checks.values()), "A hard data-quality validation failed"

validation_rows = [
    {"check": name, "status": "Pass" if passed else "Fail", "action": "Retain"}
    for name, passed in hard_checks.items()
]
validation_rows.append({
    "check": "Exact duplicate source rows",
    "status": "Issue found: %d" % duplicate_rows,
    "action": "Remove before target engineering and splitting",
})
validation_rows.append({
    "check": "Rows with at least one 1.5-IQR outlier flag",
    "status": "%d (%.1f%%)" % (rows_with_iqr_flags,
                                100 * rows_with_iqr_flags / len(df)),
    "action": "Retain; plausible and not verified as errors",
})
validation_rows.append({
    "check": "Observed quality levels",
    "status": ", ".join(map(str, sorted(df["quality"].unique()))),
    "action": "Retain; levels are inside documented range",
})
data_quality = pd.DataFrame(validation_rows)
data_quality.to_csv(OUTPUT_DIR / "data_quality_v6.csv", index=False)
outlier_summary.to_csv(OUTPUT_DIR / "outlier_summary_v6.csv")

print("Raw rows:", len(df), "| exact duplicates:", duplicate_rows)
display(data_quality)
display(outlier_summary[
    ["flagged_values", "observed_min", "observed_max"]
].round(3))

fig, axes = plt.subplots(3, 4, figsize=(12, 8))
for ax, feature in zip(axes.flat, source_features):
    sns.boxplot(x=df[feature], ax=ax, color="#6a9c78", fliersize=2,
                linewidth=0.8)
    ax.set_title(feature, fontsize=9)
    ax.set_xlabel("")
for ax in axes.flat[len(source_features):]:
    ax.axis("off")
fig.suptitle("IQR outlier audit - source laboratory measurements", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "v6_outlier_boxplots.png", dpi=120, bbox_inches="tight")
plt.show()

### 2.4 Target and duplicate issue